# Multi-Scale CNN Architecture Test

This notebook tests the multi-scale CNN architecture with parallel convolutional branches.

**Architecture Overview:**
- Multiple parallel Conv1d branches with different kernel sizes (e.g., 3, 5, 7)
- Each branch captures patterns at different temporal scales
- Outputs concatenated along channel dimension
- Optional: Stack multiple multi-scale blocks with downsampling

**Expected Benefits:**
- Better feature extraction from multiple temporal resolutions
- Similar to Inception architecture for time series
- +2-4% F1 improvement expected

In [17]:
# Imports
import sys
import os

import torch
import pytorch_lightning as L
from GradientGang.Pipeline.Architectures.Direct import Direct
from GradientGang.Pipeline.DataLoader.DataLoader import DataModule
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

## 1. Load Data

In [18]:
data_params = {
    'data_dir': "../dataset/PirateProcessed/",
    'train_file_name': "pirate_pain_train.csv",
    'train_file_name_labels': "pirate_pain_train_labels.csv",
    'test_file_name': "pirate_pain_test.csv",
    'train_global_features_file': "train_global_features.csv",
    'test_global_features_file': "test_global_features.csv",
    'batch_size': 32,
    'num_workers': 0,
    'val_split': 0.1,
    'shuffle': True,
}

# Create DataModule
data_module = DataModule(data_params)
data_module.setup()

print(f"Train samples: {len(data_module.train_dataset)}")
print(f"Val samples: {len(data_module.val_dataset)}")
print(f"Test samples: {len(data_module.test_dataset)}")

Train samples: 1919
Val samples: 66
Test samples: 1324


## 2. Test Single Multi-Scale Block

First, let's test a single multi-scale CNN block with 3 parallel branches.

In [19]:
# Architecture with single multi-scale block
params_single = {
    "EncoderParams": {
        "activation_function": "ReLU",
        "layer_type": [
            {
                "name": "MultiScaleCNN",
                "params": {
                    "in_channels": 34,  # Input channels (time series features)
                    "branch_channels": 64,  # Channels per branch
                    "kernel_sizes": [3, 5, 7],  # Multi-scale kernels
                    "use_dilation": False,  # Use actual kernel sizes
                    "pooling_type": "max",  # MaxPool after concat
                },
            },
            # After multi-scale: 192 channels (64*3), seq_len=80 (160/2 from pooling)
            {
                "name": "Flatten",
                "params": {},
            },
        ],
    },
    "GlobalFFEncoderParams": {
        "activation_function": "ReLU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {"in_features": 32, "out_features": 64, "bias": True},
            },
        ],
    },
    "FeedForwardParams": {
        "activation_function": "ReLU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {
                    "in_features": 192 * 80 + 64,  # Flattened encoder + global features
                    "out_features": 256,
                    "bias": True,
                },
            },
            {
                "name": "Linear",
                "params": {"in_features": 256, "out_features": 128, "bias": True},
            },
        ],
    },
    "OutputDim": 3,
    "LearningRate": 0.001,
    "RegularizationWeight": 0.0001,
    "ClassWeightsPath": "../dataset/PirateProcessed/class_weights.yaml",
    "SchedulerType": "CosineAnnealing",
    "T_max": 50,
    "eta_min": 1e-6,
}

# Create model
model_single = Direct(params_single)

# Test forward pass
batch = next(iter(data_module.train_dataloader()))
(time_series, global_features), labels = batch

print(f"Input time series shape: {time_series.shape}")
print(f"Input global features shape: {global_features.shape}")

with torch.no_grad():
    output = model_single((time_series, global_features))  # Pass as separate args, not tuple
    print(f"Output shape: {output.shape}")
    print(f"Output logits: {output[0]}")

print("\n✅ Single multi-scale block test passed!")

Input time series shape: torch.Size([32, 34, 160])
Input global features shape: torch.Size([32, 32])
Output shape: torch.Size([32, 3])
Output logits: tensor([-0.1374, -0.0910, -0.1523])

✅ Single multi-scale block test passed!


## 3. Test Stacked Multi-Scale Blocks

Now let's test stacking 2 multi-scale blocks with channel progression.

In [20]:
# Architecture with 2 stacked multi-scale blocks
params_stacked = {
    "EncoderParams": {
        "activation_function": "ReLU",
        "layer_type": [
            # Block 1: 34 -> 192 channels (64*3), 160 -> 80 timesteps
            {
                "name": "MultiScaleCNN",
                "params": {
                    "in_channels": 34,
                    "branch_channels": 64,
                    "kernel_sizes": [3, 5, 7],
                    "use_dilation": False,
                    "pooling_type": "max",  # Downsample by 2
                },
            },
            # Block 2: 192 -> 384 channels (128*3), 80 -> 40 timesteps
            {
                "name": "MultiScaleCNN",
                "params": {
                    "in_channels": 192,  # Output from previous block
                    "branch_channels": 128,
                    "kernel_sizes": [3, 5, 7],
                    "use_dilation": False,
                    "pooling_type": "max",  # Downsample by 2
                },
            },
            {
                "name": "Flatten",
                "params": {},
            },
        ],
    },
    "GlobalFFEncoderParams": {
        "activation_function": "ReLU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {"in_features": 32, "out_features": 64, "bias": True},
            },
        ],
    },
    "FeedForwardParams": {
        "activation_function": "ReLU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {
                    "in_features": 384 * 40 + 64,  # Flattened encoder + global features
                    "out_features": 512,
                    "bias": True,
                },
            },
            {
                "name": "Linear",
                "params": {"in_features": 512, "out_features": 128, "bias": True},
            },
        ],
    },
    "OutputDim": 3,
    "LearningRate": 0.001,
    "RegularizationWeight": 0.0001,
    "ClassWeightsPath": "../dataset/PirateProcessed/class_weights.yaml",
    "SchedulerType": "CosineAnnealing",
    "T_max": 50,
    "eta_min": 1e-6,
}

# Create model
model_stacked = Direct(params_stacked)

# Test forward pass
with torch.no_grad():
    output = model_stacked((time_series, global_features))
    print(f"Output shape: {output.shape}")
    print(f"Output logits: {output[0]}")

print("\n✅ Stacked multi-scale blocks test passed!")

Output shape: torch.Size([32, 3])
Output logits: tensor([-0.0130, -0.0734,  0.0801])

✅ Stacked multi-scale blocks test passed!

Output logits: tensor([-0.0130, -0.0734,  0.0801])

✅ Stacked multi-scale blocks test passed!


## 4. Test Dilation Mode

Test using dilated convolutions instead of larger kernels.

In [21]:
# Architecture with dilation
params_dilation = {
    "EncoderParams": {
        "activation_function": "ReLU",
        "layer_type": [
            {
                "name": "MultiScaleCNN",
                "params": {
                    "in_channels": 34,
                    "branch_channels": 64,
                    "kernel_sizes": [3, 5, 7],  # Used as dilation rates
                    "use_dilation": True,  # Use dilation instead
                    "pooling_type": "max",
                },
            },
            {
                "name": "Flatten",
                "params": {},
            },
        ],
    },
    "GlobalFFEncoderParams": {
        "activation_function": "ReLU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {"in_features": 32, "out_features": 64, "bias": True},
            },
        ],
    },
    "FeedForwardParams": {
        "activation_function": "ReLU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {
                    "in_features": 192 * 80 + 64,
                    "out_features": 256,
                    "bias": True,
                },
            },
            {
                "name": "Linear",
                "params": {"in_features": 256, "out_features": 128, "bias": True},
            },
        ],
    },
    "OutputDim": 3,  # Fixed: 3 classes (0, 1, 2)
    "LearningRate": 0.001,
    "RegularizationWeight": 0.0001,
    "ClassWeightsPath": "../dataset/PirateProcessed/class_weights.yaml",
    "SchedulerType": "CosineAnnealing",
    "T_max": 50,
    "eta_min": 1e-6,
}

# Create model
model_dilation = Direct(params_dilation)

# Test forward pass
with torch.no_grad():
    output = model_dilation((time_series, global_features))
    print(f"Output shape: {output.shape}")
    print(f"Output logits: {output[0]}")

print("\n✅ Dilation mode test passed!")

Output shape: torch.Size([32, 3])
Output logits: tensor([ 0.0172, -0.1618, -0.1476])

✅ Dilation mode test passed!
Output logits: tensor([ 0.0172, -0.1618, -0.1476])

✅ Dilation mode test passed!


## 7. Quick Direct Model Training Test

Run a quick training test with the Direct architecture to verify everything works end-to-end.

## 5. Test Autoencoder with Multi-Scale CNN

Test the autoencoder architecture with multi-scale encoder and symmetric decoder.

In [33]:
from GradientGang.Pipeline.Architectures.LightningAutoencoder import LightningAutoencoder

# Autoencoder with single multi-scale block
params_autoencoder = {
    "EncoderParams": {
        "activation_function": "ReLU",
        "layer_type": [
            {
                "name": "MultiScaleCNN",
                "params": {
                    "in_channels": 34,
                    "branch_channels": 64,
                    "kernel_sizes": [3, 5, 7],
                    "use_dilation": False,
                    "pooling_type": "max",
                },
            },
            # After multi-scale: 192 channels, seq_len=80
            {
                "name": "AdaptiveAvgPool1d",
                "params": {
                    "output_size": 1,
                },
            },
            {
                "name": "Flatten",
                "params": {},
            },
        ],
    },
    "DecoderParams": {
        "activation_function": "ReLU",
        "layer_type": [
            # Unflatten from 192 -> (192, 1)
            {
                "name": "Unflatten",
                "params": {
                    "dim": 1,
                    "unflattened_size": (192, 1),
                },
            },
            # Upsample 1 -> 20 using ConvTranspose1d (stride=20)
            {
                "name": "ConvTranspose1d",
                "params": {
                    "in_channels": 192,
                    "out_channels": 128,
                    "kernel_size": 20,
                    "stride": 20,
                    "padding": 0,
                    "bias": True,
                },
            },
            # Upsample 20 -> 160 using ConvTranspose1d (stride=8)
            {
                "name": "ConvTranspose1d",
                "params": {
                    "in_channels": 128,
                    "out_channels": 64,
                    "kernel_size": 8,
                    "stride": 8,
                    "padding": 0,
                    "bias": True,
                },
            },
            # Final conv to reconstruct original 34 channels
            {
                "name": "Conv1d",
                "params": {
                    "in_channels": 64,
                    "out_channels": 34,
                    "kernel_size": 3,
                    "padding": 1,
                    "stride": 1,
                    "dilation": 1,
                    "groups": 1,
                    "bias": True,
                    "padding_mode": "zeros",
                },
            },
        ],
    },
    "GlobalFFEncoderParams": {
        "activation_function": "ReLU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {"in_features": 32, "out_features": 64, "bias": True},
            },
        ],
    },
    "GlobalFFDecoderParams": {
        "activation_function": "ReLU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {"in_features": 64, "out_features": 32, "bias": True},
            },
        ],
    },
    "FeedForwardParams": {
        "activation_function": "ReLU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {
                    "in_features": 192 + 64,  # Encoder output + global features
                    "out_features": 256,
                    "bias": True,
                },
            },
            {
                "name": "Linear",
                "params": {"in_features": 256, "out_features": 128, "bias": True},
            },
        ],
    },
    "OutputDim": 3,
    "LearningRate": 0.001,
    "RegularizationWeight": 0.0001,
    "ReconstructionWeight": 0.5,  # Weight for reconstruction loss
    "ClassWeightsPath": "../dataset/PirateProcessed/class_weights.yaml",
    "SchedulerType": "CosineAnnealing",
    "T_max": 50,
    "eta_min": 1e-6,
    "ReconstructionLossWeight":0.1
}

# Create autoencoder model
model_autoencoder = LightningAutoencoder(params_autoencoder)

# Test forward pass
print("Testing Autoencoder forward pass...")
with torch.no_grad():
    output, reconstruction = model_autoencoder((time_series, global_features))
    print(f"Classification output shape: {output.shape}")
    print(f"Reconstruction shape time: {reconstruction[0].shape}")
    print(f"Reconstruction shape global: {reconstruction[1].shape}")
    print(f"Original time series shape: {time_series.shape}")
    print(f"Classification logits: {output[0]}")
    
    # Check reconstruction quality
    recon_mse = torch.nn.functional.mse_loss(reconstruction[0], time_series)
    print(f"Reconstruction MSE (random init): {recon_mse.item():.4f}")

print("\n✅ Autoencoder multi-scale test passed!")

Testing Autoencoder forward pass...
Classification output shape: torch.Size([32, 3])
Reconstruction shape time: torch.Size([32, 34, 160])
Reconstruction shape global: torch.Size([32, 32])
Original time series shape: torch.Size([32, 34, 160])
Classification logits: tensor([ 0.0093, -0.1760,  0.1826])
Reconstruction MSE (random init): 0.7293

✅ Autoencoder multi-scale test passed!


## 6. Quick Autoencoder Training Test

Train the autoencoder for a few epochs to verify reconstruction + classification works.

In [ ]:
# Create a fresh autoencoder for training
model_autoencoder_train = LightningAutoencoder(params_autoencoder)

# Setup callbacks
early_stopping_ae = EarlyStopping(
    monitor="val_loss",
    patience=5,
    mode="min",
    verbose=True,
)

checkpoint_ae = ModelCheckpoint(
    monitor="val_F1",
    mode="max",
    save_top_k=1,
    filename="multiscale-autoencoder-{epoch:02d}-{val_F1:.3f}",
)

# Setup logger
logger_ae = TensorBoardLogger("lightning_logs", name="multiscale_autoencoder_test")

# Create trainer
trainer_ae = L.Trainer(
    max_epochs=10,
    callbacks=[early_stopping_ae, checkpoint_ae],
    logger=logger_ae,
    accelerator="auto",
    devices=1,
    enable_progress_bar=True,
)

# Train
print("Starting autoencoder training test...")
trainer_ae.fit(model_autoencoder_train, data_module)

# Evaluate
print("\nEvaluating autoencoder on validation set...")
val_results_ae = trainer_ae.validate(model_autoencoder_train, data_module)
print(f"Validation F1: {val_results_ae[0]['val_F1']:.4f}")
print(f"Validation Classification Loss: {val_results_ae[0]['val_prediction_loss']:.4f}")
print(f"Validation Reconstruction Loss: {val_results_ae[0]['val_reconstruction_loss']:.4f}")
print(f"Validation Total Loss: {val_results_ae[0]['val_loss']:.4f}")

print("\n✅ Autoencoder training test completed successfully!")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
TPU available: False, using: 0 TPU cores


Starting autoencoder training test...



  | Name                       | Type              | Params | Mode 
-------------------------------------------------------------------------
0 | encoder                    | Encoder           | 33.0 K | train
1 | decoder                    | Decoder           | 563 K  | train
2 | globalff_encoder           | FeedForward       | 2.1 K  | train
3 | globalff_decoder           | FeedForward       | 2.1 K  | train
4 | feedforward                | FeedForward       | 99.1 K | train
5 | f1Function                 | MulticlassF1Score | 0      | train
6 | reconstructionLossFunction | MSELoss           | 0      | train
7 | predictionLossFunction     | CrossEntropyLoss  | 0      | train
-------------------------------------------------------------------------
700 K     Trainable params
0         Non-trainable params
700 K     Total params
2.800     Total estimated model params size (MB)
48        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Polimi\Master\3sem\ANN_challenges\GradientGang\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Polimi\Master\3sem\ANN_challenges\GradientGang\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


...


Evaluating autoencoder on validation set...


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃            Validate metric             ┃              DataLoader 0              ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│                 val_F1                 │           0.8030303120613098           │
│                val_loss                │           0.888991117477417            │
│          val_prediction_loss           │           0.9254295229911804           │
│        val_reconstruction_loss         │           0.5610451102256775           │
│ val_reconstruction_loss_globalFeatures │          0.13833759725093842           │
│   val_reconstruction_loss_timeSeries   │           0.4227074682712555           │
└────────────────────────────────────────┴────────────────────────────────────────┘

Validation F1: 0.8030


KeyError: 'val_classification_loss'

In [35]:
# Use the single multi-scale block model for quick test
model_test = Direct(params_single)

# Setup callbacks
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    mode="min",
    verbose=True,
)

checkpoint = ModelCheckpoint(
    monitor="val_F1",
    mode="max",
    save_top_k=1,
    filename="multiscale-{epoch:02d}-{val_F1:.3f}",
)

# Setup logger
logger = TensorBoardLogger("lightning_logs", name="multiscale_test")

# Create trainer
trainer = L.Trainer(
    max_epochs=10,
    callbacks=[early_stopping, checkpoint],
    logger=logger,
    accelerator="auto",
    devices=1,
    enable_progress_bar=True,
)

# Train
print("Starting quick training test...")
trainer.fit(model_test, data_module)

# Evaluate
print("\nEvaluating on validation set...")
val_results = trainer.validate(model_test, data_module)
print(f"Validation F1: {val_results[0]['val_F1']:.4f}")
print(f"Validation Loss: {val_results[0]['val_loss']:.4f}")

print("\n✅ Training test completed successfully!")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
TPU available: False, using: 0 TPU cores


Starting quick training test...



  | Name                   | Type              | Params | Mode 
---------------------------------------------------------------------
0 | encoder                | Encoder           | 33.0 K | train
1 | globalff_encoder       | FeedForward       | 2.1 K  | train
2 | feedforward            | FeedForward       | 4.0 M  | train
3 | f1Function             | MulticlassF1Score | 0      | train
4 | predictionLossFunction | CrossEntropyLoss  | 0      | train
---------------------------------------------------------------------
4.0 M     Trainable params
0         Non-trainable params
4.0 M     Total params
16.069    Total estimated model params size (MB)
37        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Polimi\Master\3sem\ANN_challenges\GradientGang\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Polimi\Master\3sem\ANN_challenges\GradientGang\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


...

Metric val_loss improved by 0.004 >= min_delta = 0.0. New best score: 1.092
`Trainer.fit` stopped: `max_epochs=10` reached.
`Trainer.fit` stopped: `max_epochs=10` reached.



Evaluating on validation set...


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val_F1           │    0.04545454680919647    │
│         val_loss          │    1.0924866199493408     │
│    val_prediction_loss    │    1.0924866199493408     │
└───────────────────────────┴───────────────────────────┘

Validation F1: 0.0455
Validation Loss: 1.0925

✅ Training test completed successfully!


## 8. Model Summary

Display model architecture and parameter count for both Direct and Autoencoder models.

In [ ]:
from torchinfo import summary

print("=" * 80)
print("Single Multi-Scale Block Model (Direct)")
print("=" * 80)
summary(model_single, input_data=[time_series, global_features], depth=3)

print("\n" + "=" * 80)
print("Stacked Multi-Scale Blocks Model (Direct)")
print("=" * 80)
summary(model_stacked, input_data=[time_series, global_features], depth=3)

print("\n" + "=" * 80)
print("Multi-Scale Autoencoder Model")
print("=" * 80)
summary(model_autoencoder, input_data=[time_series, global_features], depth=3)

# Count parameters
single_params = sum(p.numel() for p in model_single.parameters())
stacked_params = sum(p.numel() for p in model_stacked.parameters())
autoencoder_params = sum(p.numel() for p in model_autoencoder.parameters())

print(f"\nParameter comparison:")
print(f"Single block (Direct): {single_params:,} parameters")
print(f"Stacked blocks (Direct): {stacked_params:,} parameters")
print(f"Autoencoder: {autoencoder_params:,} parameters")
print(f"\nStacked vs Single increase: {(stacked_params / single_params - 1) * 100:.1f}%")
print(f"Autoencoder vs Single increase: {(autoencoder_params / single_params - 1) * 100:.1f}%")

## Summary

This notebook successfully tested the multi-scale CNN architecture:

✅ **Single multi-scale block**: 3 parallel branches with kernels [3, 5, 7]
✅ **Stacked multi-scale blocks**: 2 layers with channel progression
✅ **Dilation mode**: Alternative to larger kernels
✅ **Autoencoder architecture**: Multi-scale encoder + symmetric decoder
✅ **End-to-end training**: Works with PyTorch Lightning for both Direct and Autoencoder

**Key Findings:**
- Multi-scale CNN successfully captures patterns at different temporal resolutions
- Autoencoder combines reconstruction and classification objectives
- Both architectures support flexible configuration via hyperparameters

**Next steps:**
1. Integrate into Optuna optimization (OptunaGeneralGlobalStudy.ipynb)
2. Search optimal hyperparameters:
   - Number of multi-scale layers (1-2)
   - Branch channels per layer
   - Kernel size combinations
   - Dilation vs standard kernels
   - Pooling types
   - Architecture type (Direct vs Autoencoder)
3. Compare against baseline Conv1d and RNN architectures